# NeuroTrace V6 — real UCI395 → GPU smoke → ONNX → locked holdout

**Research only, not clinical validation.** Start in a fresh Colab GPU runtime
(A100/T4),or Kaggle GPU runtime with Internet. No manually constructed records or
partition required: the loader builds both from original recorded text files.

Default: real-only Wacom baseline,2 supervised epochs per fold,no SSL. Flip
`SMOKE=False` only after this complete smoke passes. P100 depends on driver support.
Synthetic pseudo-PD requires explicit opt-in and real authorized HC touch/mouse;
it is never evidence of mobile-PD sensitivity. No push/merge or V5 artifact changes.

**Source correction:** verified UCI mirror is headerless seven-column
X,Y,Z,Pressure,GripAngle,Timestamp,TestID. Explicit named six-column files also work,
but are not the format verified in that mirror. GripAngle is NOT called azimuth.
README does not state clock units: `milliseconds` below is an exposed assumption,
not inferred from a guessed rate. Source x/y calibration is unknown. Missing age
is schema integer -1; missing UPDRS/azimuth is null/NaN,not a fabricated measurement.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def run_live(command):
    print('$', ' '.join(map(str,command)), flush=True)
    env=dict(os.environ, PYTHONUNBUFFERED='1')
    env.setdefault('OMP_NUM_THREADS','2'); env.setdefault('MKL_NUM_THREADS','2')
    with subprocess.Popen(list(map(str,command)),stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT,text=True,bufsize=1,env=env) as process:
        for line in process.stdout: print(line,end='',flush=True)
        code=process.wait()
    if code: raise subprocess.CalledProcessError(code,command)

REPO_DIR=Path('/kaggle/working/NeuroTrace-AI' if Path('/kaggle').exists() else '/content/NeuroTrace-AI')
BRANCH='arena/4edc99ce-neurotrace-ai'
if not (REPO_DIR/'.git').exists():
    run_live(['git','clone','--branch',BRANCH,'https://github.com/VaKhang0709/NeuroTrace-AI.git',REPO_DIR])
os.chdir(REPO_DIR)
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip()==BRANCH
# Fast-forward only on this same branch. Never discard local user modifications.
run_live(['git','fetch','origin',BRANCH])
run_live(['git','merge','--ff-only','origin/'+BRANCH])
print('Exact code commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

$ git fetch origin arena/4edc99ce-neurotrace-ai
From https://github.com/VaKhang0709/NeuroTrace-AI
 * branch            arena/4edc99ce-neurotrace-ai -> FETCH_HEAD
   2d94669..8731838  arena/4edc99ce-neurotrace-ai -> origin/arena/4edc99ce-neurotrace-ai
$ git merge --ff-only origin/arena/4edc99ce-neurotrace-ai
Updating 2d94669..8731838
Fast-forward
 BLOCKERS.md                                    |  26 +
 README.md                                      |  54 +-
 docs/v6_gpu_execution_guide.md                 | 446 +++++++--------
 notebooks/v6_gpu_beast_training.ipynb          | 303 ++++++----
 reports/v6_correctness_report.md               |  15 +
 reports/v6_gpu_workflow_verification.md        |  15 +
 reports/v6_uci395_cpu_execution_summary.json   |  23 +
 reports/v6_uci395_loader_verification.md       | 177 ++++++
 scripts/train_v6_gpu_standalone.py             | 155 +++--
 src/data_loaders/public_trajectory_parser.py   |  37 +-
 src/data_loaders/uci395_loader.py              | 240 ++++

## Install before importing torch

Use a fresh runtime. If torch was already imported, restart after installation.
Downloads may be several GB. No credentials belong in notebook cells or outputs.

In [ ]:
run_live([sys.executable,'-m','pip','install','-r','requirements-v6-gpu.txt'])
import torch
from scripts.train_v6_gpu_standalone import gpu_info
gpu_info(require=True)
print('cuDNN:',torch.backends.cudnn.version())
# Batch2 is a conservative starting point,not measured VRAM guidance.

$ /usr/bin/python3 -m pip install -r requirements-v6-gpu.txt
{
  "name": "Tesla T4",
  "vram_gib": 14.56,
  "capability": [
    7,
    5
  ],
  "torch": "2.5.1+cu124"
}
cuDNN: 90100


## Real local recordings or automatic official UCI download

Set `LOCAL_UCI` to an extracted directory or authorized original zip. Kaggle
conventional `control/` + `parkinson/` folders are auto-detected if unambiguous.
Otherwise the official UCI zip is downloaded automatically. On network denial,set
the local path and rerun; there is NO generated/mock replacement.

For private mobile recordings mount authorized storage separately (optional Drive
mount below); do not publish credentials,participant IDs or raw paths. Human
identity linkage across devices still needs review. Do not substitute drawing IDs
for participant IDs. A **new** partition is deterministically stratified by true
parent label; existing locked partitions are never silently overwritten.

In [ ]:
# OPTIONAL Colab storage (operator authentication only,no embedded credentials):
# from google.colab import drive
# drive.mount('/content/drive')
LOCAL_UCI=None  # e.g. '/kaggle/input/your-dataset/hw_dataset' or '/content/uci.zip'
if LOCAL_UCI is None and Path('/kaggle/input').exists():
    roots={p.parent.parent for p in Path('/kaggle/input').rglob('*.txt')
           if p.parent.name.lower()=='control'
           and (p.parent.parent/'parkinson').is_dir()}
    if len(roots)==1: LOCAL_UCI=next(iter(roots))
    else: print('No unique conventional UCI folder detected; official download will be attempted.')
RUN_ROOT=Path('data/processed/v6_uci395'); RUN_ROOT.mkdir(parents=True,exist_ok=True)
RECORDS=RUN_ROOT/'v6_unified_records.json' # created by loader if absent
PARTITION=RUN_ROOT/'partition.json'       # locked after creation
PREPARED=RUN_ROOT/'v6_prepared_records.json'
FEATURE_CONTRACT='v6-dwt19-v1'
UCI_TIME_UNIT='milliseconds' # explicit assumption if source header has no unit; never guess Hz
UCI_AZIMUTH_UNIT='unknown'   # raw7 GripAngle is NOT azimuth
SMOKE=False
EPOCHS,SSL_EPOCHS,BATCH_SIZE=(2,0,2) if SMOKE else (200,100,2)
USE_SYNTHETIC=False # explicit experimental pseudo-label ablation only
PER_DOMAIN=5001
OUTPUT=Path('models/v6/smoke_checkpoints' if SMOKE else 'models/v6/checkpoints')
EXPORT=Path('models/v6/smoke_export' if SMOKE else 'models/v6/export')
METRICS=Path('reports/private/v6_smoke_holdout.md' if SMOKE else 'reports/private/v6_holdout.md')
print('Local input:',LOCAL_UCI or 'official download','; smoke:',SMOKE)

No unique conventional UCI folder detected; official download will be attempted.
Local input: official download ; smoke: True


## Real conversion + feature QC BEFORE training

This creates missing records AND partition before attempting to read them. It
prints file/row skip reasons,duplicate-last removals,clock assumptions,HC/PD counts,
locked partition sizes,and the actual prepared [N,19,512] tensor count. No resampling
in the raw loader: the feature pipeline alone resamples and rejects short trials
without stretching time. Feature QC occurs before creating a new locked partition.

With Wacom-only UCI,synthetic generation is **skipped** by default. Do not enable
the experimental ablation without authorized real touch/mouse HC sources and
fold-scoped recorded PD donors. Failed preparation stops the notebook visibly.

In [ ]:
common=[sys.executable,'-m','scripts.train_v6_gpu_standalone',
        '--records',str(RECORDS),'--partition',str(PARTITION),
        '--prepared-records',str(PREPARED),'--output',str(OUTPUT),
        '--feature-contract',FEATURE_CONTRACT,'--uci-time-unit',UCI_TIME_UNIT,
        '--uci-azimuth-unit',UCI_AZIMUTH_UNIT,'--per-domain',str(PER_DOMAIN),
        '--epochs',str(EPOCHS),'--ssl-epochs',str(SSL_EPOCHS),'--batch-size',str(BATCH_SIZE)]
if LOCAL_UCI: common+=['--local-uci-dir',str(LOCAL_UCI)]
if USE_SYNTHETIC: common+=['--use-synthetic','--synthetic-fraction','0.25']
run_live(common+['--prepare-only'])
import json,numpy as np
partition=json.loads(PARTITION.read_text())
print('LOCKED subject partition:',{k:len(v) for k,v in partition.items()})
with np.load(PREPARED.with_suffix('.features.npz'),allow_pickle=False) as z:
    print('ACTUAL real feature cache:',z['X'].shape,'finite:',np.isfinite(z['X']).all())

$ /usr/bin/python3 -m scripts.train_v6_gpu_standalone --records data/processed/v6_uci395/v6_unified_records.json --partition data/processed/v6_uci395/partition.json --prepared-records data/processed/v6_uci395/v6_prepared_records.json --output models/v6/smoke_checkpoints --feature-contract v6-dwt19-v1 --uci-time-unit milliseconds --uci-azimuth-unit unknown --per-domain 5001 --epochs 2 --ssl-epochs 0 --batch-size 2 --prepare-only
{
  "name": "Tesla T4",
  "vram_gib": 14.56,
  "capability": [
    7,
    5
  ],
  "torch": "2.5.1+cu124"
}
{
  "raw_layout": "verified headerless X,Y,Z,Pressure,GripAngle,Timestamp,TestID; named headers explicitly mapped",
  "time_unit": "milliseconds",
  "time_scale": 0.001,
  "time_unit_evidence": "CONFIGURED ASSUMPTION: mirrored README does not declare timestamp units; no rate inference",
  "azimuth_unit": "unknown",
  "coordinate_units": "unknown (source README does not calibrate native coordinates)",
  "counters": {
    "total_txt_files": 79,
    "layout_v

## Five-fold GPU training (default:2-epoch smoke,not performance evidence)

Full15.78M-parameter model,CUDA AMP,gradient accumulation target256,five-epoch
warmup/cosine restarts. Smoke2epochs intentionally ends during warmup; it is an
execution check,NOT a trained clinical model. Full-run SSL is fold-scoped.

Every epoch's loss/validation metrics stream below as well as to private CSV.
Selection/calibration uses real validation subjects only; locked holdout never
enters SSL,augmentation donors,selection or calibration. Save several GB privately.
UCI is Wacom-only. No notebook step validates mobile PD or fixes the healthy-phone
high-PD web-domain gap.

In [ ]:
run_live(common)
checkpoints=[str(OUTPUT/f'fold_{fold}_best.pth') for fold in range(5)]
assert all(Path(p).is_file() for p in checkpoints)
print('Five fold checkpoints created. Smoke scores are not clinical evidence.')

## Member ONNX export + validation-only calibration

Selects best member by validation AUC, not holdout. Exports dynamic batch/time,
checks100tensors across lengths256/384/512/768. Real training remains512steps.
Isotonic calibration is AFTER subject-mean probabilities; validation-selected Youden
threshold is a V6-only research operating point, not a clinical optimum. Use
`fixed_0.5` for a prespecified fixed threshold. Scaler JSON clearly distinguishes
measured training statistics from per-window normalization: **do not apply global
mean/std a second time**. No changes to V5 root artifacts.

In [ ]:
run_live([sys.executable,'-m','scripts.export_v6_onnx_pipeline',
          '--checkpoints',*checkpoints,'--records',str(PREPARED),'--partition',str(PARTITION),
          '--output',str(EXPORT),'--threshold-policy','fixed_0.5'])
assert (EXPORT/'neurotrace_v6_tremorformer.onnx').is_file()
print('Parity manifest:',(EXPORT/'neurotrace_v6_tremorformer.parity.json').read_text())

## Locked holdout subject metrics CSV (no threshold search)

Raw five-member/ten-variant ensemble probabilities at0.50. Selected-member
calibration/threshold are NOT silently applied to the ensemble. Reports aggregate
by true subject and include domain-specific estimates and confidence intervals.
The CSV below is **subject-primary aggregate metrics**,not publicly exposed patient
predictions. Private NPZ keeps necessary per-parent audit information.

Wacom-only estimates do not measure touch/mobile PD sensitivity. Missing mobile
PD remains UNMEASURABLE; healthy-phone high-PD web behavior remains unresolved.
A2epoch smoke is only workflow verification,no clinical/safety claim.

In [ ]:
PREDICTIONS=Path('reports/private/v6_smoke_predictions.npz' if SMOKE else 'reports/private/v6_predictions.npz')
run_live([sys.executable,'-m','src.inference.v6_ensemble_inference',
          '--records',str(PREPARED),'--partition',str(PARTITION),'--checkpoints',*checkpoints,
          '--output',str(PREDICTIONS)])
run_live([sys.executable,'-m','src.evaluation.v6_eval','--predictions',str(PREDICTIONS),
          '--partition',str(PARTITION),'--output',str(METRICS)])
assert METRICS.with_suffix('.csv').is_file()
print('Subject-primary CSV:',METRICS.with_suffix('.csv'))
print(METRICS.with_suffix('.csv').read_text())

## Preserve outputs / review

Copy `models/v6/`, private fold manifests and reports to your authorized Drive or
Kaggle private output storage. Do NOT upload raw trajectories, credentials, patient
IDs or unreviewed clinical predictions to public repositories. This notebook has no
`git push` or merge step. Tech Lead reviews PR #4 manually. Review
`docs/v6_gpu_execution_guide.md` and `BLOCKERS.md` before any model promotion.